# Solution B3 · Polynomial pipeline with Ridge and GridSearchCV

**Type**: Implement with scikit-learn

## Objective

Build a `Pipeline` (PolynomialFeatures, StandardScaler, Ridge), choose degree and regularization with
`GridSearchCV`, and measure what happens outside the range of the data with a degree that is too high.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import GridSearchCV, KFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

d = ds.energy_consumption()
X, y = d[["temperature"]], d["consumption"]
t45 = pd.DataFrame({"temperature": [45.0]})
true_45 = 50 + 0.16 * (45 - 17) ** 2               # the true curve, which the model does not know
print("true value at 45 degrees:", round(true_45, 1), "kWh")

## Commented solution

In [ ]:
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=0)
pipe = Pipeline([("poly", PolynomialFeatures(include_bias=False)),
                 ("scaler", StandardScaler()),
                 ("ridge", Ridge())])

The step names serve to address the parameters: `poly__degree` means "the `degree` parameter of the `poly`
step". The scaler sits **after** `PolynomialFeatures`, because it is the high powers that have enormous scales.

In [ ]:
grid = {"poly__degree": list(range(1, 11)), "ridge__alpha": np.logspace(-4, 2, 7)}
search = GridSearchCV(pipe, grid, cv=KFold(5, shuffle=True, random_state=0),
                      scoring="neg_root_mean_squared_error")
search.fit(X_tr, y_tr)

chosen_degree = search.best_params_["poly__degree"]
chosen_alpha = search.best_params_["ridge__alpha"]
rmse_cv = -search.best_score_
rmse_test = mean_squared_error(y_te, search.predict(X_te), squared=False)
print("degree", chosen_degree, "| alpha", chosen_alpha, "| RMSE cv", round(rmse_cv, 2), "| RMSE test", round(rmse_test, 2))

In [ ]:
linear = LinearRegression().fit(X_tr, y_tr)
poly10 = Pipeline([("poly", PolynomialFeatures(10, include_bias=False)),
                   ("scaler", StandardScaler()),
                   ("lr", LinearRegression())]).fit(X_tr, y_tr)
rmse_test_linear = mean_squared_error(y_te, linear.predict(X_te), squared=False)
rmse_test_poly10 = mean_squared_error(y_te, poly10.predict(X_te), squared=False)
print(f"RMSE test: chosen {rmse_test:.2f} | linear {rmse_test_linear:.2f} | degree 10 {rmse_test_poly10:.2f}")

In [ ]:
prediction_45_chosen = float(search.predict(t45)[0])
prediction_45_linear = float(linear.predict(t45)[0])
prediction_45_poly10 = float(poly10.predict(t45)[0])
print(f"at 45 degrees -> true {true_45:.0f} | chosen {prediction_45_chosen:.0f} | "
      f"linear {prediction_45_linear:.0f} | degree 10 {prediction_45_poly10:.0f}")

## Check

In [ ]:
with verify():
    check(len(X_te) == 48, "70/30 split", "X_te should have 48 rows (30% of 160)")
    check(2 <= chosen_degree <= 5, "the search chooses a low degree (the true curve is quadratic)",
          "the chosen degree should be between 2 and 5: check the scoring and the grid")
    check(rmse_cv > 0, "rmse_cv is positive", "use the minus sign: best_score_ is a negative score")
    check(close(rmse_test, mean_squared_error(y_te, search.predict(X_te), squared=False)),
          "rmse_test is computed on the test set", "rmse_test must be the RMSE of search.predict(X_te)")
    check(rmse_test_linear > 2 * rmse_test, "the line is much worse than the chosen model (the trend is U-shaped)",
          "the linear model should have an RMSE at least double")
    check(rmse_test_poly10 < 1.3 * rmse_test,
          "the degree 10 polynomial has an almost equal test error: INSIDE the range you don't see the problem",
          "degree 10 with the scaler should have a comparable test error: check that you used the scaler")
    check(abs(prediction_45_chosen - true_45) < 20, "the chosen model extrapolates reasonably",
          "the prediction at 45 degrees of the chosen model should be close to the true value")
    check(abs(prediction_45_poly10) > 1000, "degree 10 explodes outside the range",
          "the degree 10 polynomial should give an absurd prediction (thousands of kWh) at 45 degrees")
done()

## What you learned

- The test error of the degree 10 polynomial is **practically equal** to that of the chosen model: if you stop
  at that number you notice nothing. Outside the range of the data it predicts thousands of kWh at 45 degrees.
- A high-degree polynomial oscillates between the points (few constraints) and diverges at the edges: the high powers dominate as soon as
  you leave the training interval. The model chosen by the search, of low degree, stays close to the true curve.
- `GridSearchCV` chooses hyperparameters based on the performance **inside** the range of the data; extrapolation has to be
  judged with a separate argument (what range do I need? what behavior is physically plausible?).

## Common pitfall

Reporting the RMSE of `search.best_score_` as the final estimate of the performance: it is the maximum among many
combinations tried on the same folds, so it is optimistic. For the honest estimate you need the test set, touched only
once (or a nested cross-validation).

## Interview question

> *Click on a question to show its answer.*

<details><summary>Your degree 10 polynomial model has an excellent test error. Why don't you trust it yet?</summary>

Because the test set has the same distribution as the training set: it measures the ability to interpolate, not to
extrapolate. A high-degree polynomial can follow the points well inside the range and diverge right outside it. I would check
whether values outside the range will arrive in production (drift), I would look at the predictions on plausible extreme inputs and
I would prefer a low-degree model with regularization.

</details>